<h3>PCA Machine Learning Tutorial Exercise Solution</h3>

Download heart disease dataset heart.csv in [Exercise](https://github.com/codebasics/py/tree/master/ML/18_PCA/Exercise) folder and do following, (credits of dataset:  https://www.kaggle.com/fedesoriano/heart-failure-prediction)

1. Load heart disease dataset in pandas dataframe
1. Remove outliers using Z score. Usual guideline is to remove anything that has Z score > 3 formula or Z score < -3
1. Convert text columns to numbers using label encoding and one hot encoding
1. Apply scaling
1. Build a classification model using various methods (SVM, logistic regression, random forest) and check which model gives you the best accuracy
1. Now use PCA to reduce dimensions, retrain your model and see what impact it has on your model in terms of accuracy. Keep in mind that many times doing PCA reduces the accuracy but computation is much lighter and that's the trade off you need to consider while building models in real life


[Solution Link](https://github.com/codebasics/py/blob/master/ML/18_PCA/Exercise/PCA_heart_disease_prediction_exercise_solution.ipynb)



 

In [1]:
import pandas as pd

df=pd.read_csv("./heart.csv")
df.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


# separate features and target


In [2]:
X=df.drop(["HeartDisease"],axis="columns")
y=df["HeartDisease"]

In [3]:
X.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up


In [4]:
y.head()

0    0
1    1
2    0
3    1
4    0
Name: HeartDisease, dtype: int64

In [5]:
X.columns

Index(['Age', 'Sex', 'ChestPainType', 'RestingBP', 'Cholesterol', 'FastingBS',
       'RestingECG', 'MaxHR', 'ExerciseAngina', 'Oldpeak', 'ST_Slope'],
      dtype='object')

In [6]:
X.describe()

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak
count,918.000000,918.000000,918.000000,918.000000,918.000000,918.000000
mean,53.510893,132.396514,198.799564,0.233115,136.809368,0.887364
std,9.432617,18.514154,109.384145,0.423046,25.460334,1.066570
min,28.000000,0.000000,0.000000,0.000000,60.000000,-2.600000
25%,47.000000,120.000000,173.250000,0.000000,120.000000,0.000000
50%,54.000000,130.000000,223.000000,0.000000,138.000000,0.600000
75%,60.000000,140.000000,267.000000,0.000000,156.000000,1.500000
max,77.000000,200.000000,603.000000,1.000000,202.000000,6.200000


# One-hot encode

In [7]:
X_dummy=pd.get_dummies(X,drop_first=True)
X_dummy.head()

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,Sex_M,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_Normal,RestingECG_ST,ExerciseAngina_Y,ST_Slope_Flat,ST_Slope_Up
0,40,140,289,0,172,0.0,True,True,False,False,True,False,False,False,True
1,49,160,180,0,156,1.0,False,False,True,False,True,False,False,True,False
2,37,130,283,0,98,0.0,True,True,False,False,False,True,False,False,True
3,48,138,214,0,108,1.5,False,False,False,False,True,False,True,True,False
4,54,150,195,0,122,0.0,True,False,True,False,True,False,False,False,True


In [8]:
from sklearn.preprocessing import StandardScaler
import numpy as np

scaler=StandardScaler()
z_scores=scaler.fit_transform(X_dummy)
threshold=3



# Keep only rows where all features have |Z| < 3

In [9]:
mask=((np.absolute(z_scores) < threshold).all(axis=1))

In [10]:

X_filtered = X_dummy[mask]
y_filtered = y[mask]

In [11]:
X_filtered

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,Sex_M,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_Normal,RestingECG_ST,ExerciseAngina_Y,ST_Slope_Flat,ST_Slope_Up
0,40,140,289,0,172,0.0,True,True,False,False,True,False,False,False,True
1,49,160,180,0,156,1.0,False,False,True,False,True,False,False,True,False
2,37,130,283,0,98,0.0,True,True,False,False,False,True,False,False,True
3,48,138,214,0,108,1.5,False,False,False,False,True,False,True,True,False
4,54,150,195,0,122,0.0,True,False,True,False,True,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
912,57,140,241,0,123,0.2,False,False,False,False,True,False,True,True,False
914,68,144,193,1,141,3.4,True,False,False,False,True,False,False,True,False
915,57,130,131,0,115,1.2,True,False,False,False,True,False,True,True,False
916,57,130,236,0,174,0.0,False,True,False,False,False,False,False,True,False


In [12]:
X_filtered.shape

(854, 15)

In [13]:
y_filtered.shape

(854,)

# IMPORTANT: Scale AGAIN after outlier removal

- Because after removing rows, the means/std changed.

In [14]:
from sklearn.preprocessing import StandardScaler

scaler2=StandardScaler()
X_scaled=scaler2.fit_transform(X_filtered)
X_scaled

array([[-1.43962926,  0.47844751,  0.84463206, ..., -0.84753163,
        -1.        ,  1.13558101],
       [-0.47502369,  1.65189432, -0.16266314, ..., -0.84753163,
         1.        , -0.88060648],
       [-1.76116445, -0.10827589,  0.78918462, ..., -0.84753163,
        -1.        ,  1.13558101],
       ...,
       [ 0.38240348, -0.10827589, -0.61548392, ...,  1.17989697,
         1.        , -0.88060648],
       [ 0.38240348, -0.10827589,  0.35484632, ..., -0.84753163,
         1.        , -0.88060648],
       [-1.65398605,  0.36110283, -0.20886935, ..., -0.84753163,
        -1.        ,  1.13558101]], shape=(854, 15))

In [15]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test=train_test_split(X_scaled,y_filtered,test_size=0.2,random_state=20)

In [16]:
len(X_train)

683

In [17]:
len(X_test)

171

# finding out the best model using GridSearchCV

In [18]:
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

model_params={
    'SVM':{
        'model':SVC(),
        "params":{
            'C':[i for i in range(20)],
            'kernel':['linear', 'poly', 'rbf', 'sigmoid'],
            'gamma':["auto"]
        }
    },
    'LogisticRegression':{
        'model':LogisticRegression(),
        'params':{
            "penalty":["l1","l2"],
            "C":[i for i in range(20)],
            "solver":['lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'],
            "multi_class":['auto', 'ovr', 'multinomial']

        },
    },
    "RandomForestClassifier":{
        "model":RandomForestClassifier(),
        "params":{
            "n_estimators":[i for i in range(100)],
            "criterion":['gini', 'entropy', 'log_loss']

        }
    }

}

In [19]:
import warnings
warnings.filterwarnings("ignore")

In [20]:
from sklearn.model_selection import GridSearchCV
scores=[]

for model_name,model_paramaters in model_params.items():
    clf=GridSearchCV(model_paramaters["model"],model_paramaters["params"],return_train_score=False)
    clf.fit(X_train,y_train)
    scores.append({
        "model":model_name,
        "best_score":clf.best_score_,
        "params":clf.best_params_
    })
    







In [21]:
df1=pd.DataFrame(scores,columns=["model","best_score","params"])
df1

,model,best_score,params
0,SVM,0.865232,"{'C': 2, 'gamma': 'auto', 'kernel': 'rbf'}"
1,LogisticRegression,0.863815,"{'C': 1, 'multi_class': 'auto', 'penalty': 'l1..."
2,RandomForestClassifier,0.882782,"{'criterion': 'gini', 'n_estimators': 73}"


In [22]:
clf.best_estimator_

,n_estimators,73
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [23]:
clf.best_params_

{'criterion': 'gini', 'n_estimators': 73}

In [24]:
clf.score(X_test,y_test)

0.9122807017543859

# use scaling

# Now using PCA

In [25]:
from sklearn.decomposition import PCA

pca=PCA(0.95)
X_train_pca=pca.fit_transform(X_train)

In [26]:
best_model=clf.best_estimator_
best_model.fit(X_train_pca,y_train)

,n_estimators,73
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [27]:
X_test_pca=pca.transform(X_test)

In [28]:
pca_accuracy=best_model.score(X_test_pca,y_test)

In [29]:
pca_accuracy

0.9122807017543859